In [ ]:
import pandas as pd
from pathlib import Path

def collect_metrics(metrics_dir=".", output_csv="metrics_summary.csv"):
    metrics_path = Path(metrics_dir)
    records = []

    for file_path in metrics_path.glob("*.txt"):
        parts = file_path.stem.split("_")
        if len(parts) < 5:
            continue

        algorithm = parts[1]
        difficulty = parts[2]
        puzzle = parts[3]

        metrics = {}
        with open(file_path) as f:
            for line in f:
                if ":" in line:
                    k, v = line.strip().split(":")
                    metrics[k.strip()] = int(v.strip())

        record = {
            "algorithm": algorithm,
            "difficulty": difficulty,
            "puzzle": puzzle,
            **metrics
        }
        records.append(record)

    df = pd.DataFrame(records)

    if not df.empty:
        difficulty_order = ["easy", "medium", "hard", "extreme"]
        df["difficulty"] = pd.Categorical(df["difficulty"], categories=difficulty_order, ordered=True)

        algorithm_order = ["bt", "fc", "ac3", "sa", "ga"]
        df["algorithm"] = pd.Categorical(df["algorithm"], categories=algorithm_order, ordered=True)

        df = df.sort_values(by=["algorithm", "difficulty", "puzzle"])
        df.to_csv(output_csv, index=False)

        for algo, group in df.groupby("algorithm"):
            print(algo)
            print(group.to_string(index=False))

    return df

if __name__ == "__main__":
    df = collect_metrics("", "metrics_summary.csv")
